# NeuronScope SDK: Kaggle quickstart

Attach a Kaggle Dataset containing the repository's `sdk/` directory, then add Kaggle Secrets named `NEURONSCOPE_PROJECT_ID`, `NEURONSCOPE_ENDPOINT`, and `NEURONSCOPE_API_KEY`. This notebook trains a small synthetic classifier and streams one metric record per epoch. It does not contain or print credentials.

In [ ]:
import importlib.util
from pathlib import Path
import subprocess
import sys

sdk_candidates = []
for dataset_root in Path("/kaggle/input").iterdir():
    if (dataset_root / "neuronscope").is_dir():
        sdk_candidates.append(dataset_root)
    elif (dataset_root / "sdk" / "neuronscope").is_dir():
        sdk_candidates.append(dataset_root / "sdk")

if not sdk_candidates:
    raise FileNotFoundError(
        "Attach a Kaggle Dataset containing the NeuronScope sdk/ folder."
    )
sys.path.insert(0, str(sdk_candidates[0]))
if importlib.util.find_spec("httpx") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "httpx"])

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from kaggle_secrets import UserSecretsClient

import neuronscope
from neuronscope import MonitorConfig

secrets = UserSecretsClient()
project_id = secrets.get_secret("NEURONSCOPE_PROJECT_ID")
endpoint = secrets.get_secret("NEURONSCOPE_ENDPOINT")
api_key = secrets.get_secret("NEURONSCOPE_API_KEY")

torch.manual_seed(7)
features = torch.randn(256, 4)
labels = (features[:, 0] + features[:, 1] > 0).long()
dataset = TensorDataset(features, labels)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)
probe_loader = DataLoader(dataset, batch_size=32, shuffle=False)

model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_function = nn.CrossEntropyLoss()
monitor = neuronscope.init(
    model=model,
    project_id=project_id,
    experiment_name="kaggle-quickstart",
    endpoint=endpoint,
    api_key=api_key,
    class_mapping={0: "negative", 1: "positive"},
    config=MonitorConfig(window_size=5),
)


In [ ]:
try:
    for epoch in range(5):
        model.train()
        for batch_features, batch_labels in train_loader:
            optimizer.zero_grad(set_to_none=True)
            loss = loss_function(model(batch_features), batch_labels)
            loss.backward()
            optimizer.step()

        monitor.log_epoch(epoch, probe_loader=probe_loader)
        print(f"Finished epoch {epoch + 1}; monitor status: {monitor.status()}")
finally:
    delivered = monitor.finish()
    print(f"Final telemetry flush succeeded: {delivered}")

The SDK retains only bounded metric/event history in process memory. To use class-wise ablation, enable it explicitly in `MonitorConfig` and select a small number of neurons; it adds evaluation forwards.